In [ ]:
import xarray as xr
import scipy
import xskillscore as xs
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs

import albedo_functions as af

In [ ]:
def p_cal(ctrl, sens):
    a = ctrl.resample(time='YE').mean().var(dim='time')
    b = sens.resample(time='YE').mean().var(dim='time')

# Add a small constant to avoid division by zero
    epsilon = 1e-8
    a += epsilon
    b += epsilon

    F = a / b
    #df = len(ctrl.resample(time='YE').mean())
    df = ctrl.resample(time='YE').mean().sizes['time']
    p_value = scipy.stats.f.cdf(F, df, df)
    p_value = xr.Dataset({
        'p': xr.DataArray(p_value, dims=('lat', 'lon'),
                          coords={'lon': a['lon'],'lat': a['lat']})
    })
    return p_value

In [ ]:
def interannual_std(dset, n_bootstrap):
    """
    Calculate interannual standard deviation and its significance via bootstrapping.
    
    Parameters:
        dset (xarray.Dataset): Dataset containing 'time' dimension.
        varname (str): Variable name inside the dataset.
        n_bootstrap (int): Number of bootstrap iterations.
    
    Returns:
        xr.DataArray: observed_std
    """
    # Step 1: Group by year
    annual_mean = dset.groupby('time.year').mean('time')
    
    # Step 2: Observed std dev
    observed_std = annual_mean.std('year')
    
    # Step 3: Bootstrap resampling using xskillscore
    resampled = xs.resampling.resample_iterations_idx(
        annual_mean, n_bootstrap, dim='year', replace=True, dim_max=annual_mean.year.size
    )
    boot_std = resampled.std(dim='year')
    return observed_std

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
BASE_PATH = '/confess/dicarlo/00-dataset/02-confess/01-confess_data/'
SAVE_PATH = '/confess/dicarlo/cartella_ordinata/'

In [ ]:
lai_hv_a52o = xr.open_dataset(BASE_PATH+exp_sens+'/original_resolution/lai_hv/'+exp_sens+'_lai_hv_199311-201910_1x1.nc', chunks=-1)['LAI_HV']
lai_lv_a52o = xr.open_dataset(BASE_PATH+exp_sens+'/original_resolution/lai_lv/'+exp_sens+'_lai_lv_199311-201910_1x1.nc', chunks=-1)['LAI_LV']
# Filter: remove low standard deviation data
lai_hv_a52o = xr.where(lai_hv_a52o.std('time') >= 0.0005, lai_hv_a52o, np.nan)
lai_lv_a52o= xr.where(lai_lv_a52o.std('time') >= 0.0005, lai_lv_a52o, np.nan)

In [ ]:
a1ua_land_file = '/confess/dicarlo/00-dataset/02-confess/03-bc/'+exp_ctrl+'/icmcl_a1ua_regular_1x1.nc'
dset_a1ua = xr.open_dataset(a1ua_land_file, chunks=-1)

lai_hv_a1ua = dset_a1ua['lai_hv'].sel(time=slice(lai_hv_a52o.time[0],lai_hv_a52o.time[-1]))
lai_lv_a1ua = dset_a1ua['lai_lv'].sel(time=slice(lai_hv_a52o.time[0],lai_hv_a52o.time[-1]))
# Filter: remove low standard deviation data
lai_hv_a1ua = xr.where(lai_hv_a1ua.std('time') >= 0.0005, lai_hv_a1ua, np.nan)
lai_lv_a1ua= xr.where(lai_lv_a1ua.std('time') >= 0.0005, lai_lv_a1ua, np.nan)

In [ ]:
# Compute interannual stds
interannual_lai_hv_ctrl_std = af.land_seas_mask(interannual_std(lai_hv_a1ua, 1000))
interannual_lai_hv_sens_std = af.land_seas_mask(interannual_std(lai_hv_a52o, 1000))
interannual_lai_lv_ctrl_std = af.land_seas_mask(interannual_std(lai_lv_a1ua, 1000))
interannual_lai_lv_sens_std = af.land_seas_mask(interannual_std(lai_lv_a52o, 1000))

In [ ]:
# Compute p-values
p_value_lai_hv = af.land_seas_mask(p_cal(lai_hv_a1ua, lai_hv_a52o))
p_value_lai_lv = af.land_seas_mask(p_cal(lai_lv_a1ua, lai_lv_a52o))

In [ ]:
# Convert to Dataset and save
interannual_lai_hv_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_lai_hv_std.nc', compute=True)
interannual_lai_hv_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_lai_hv_std.nc', compute=True)
interannual_lai_lv_ctrl_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a1ua_lai_lv_std.nc', compute=True)
interannual_lai_lv_sens_std.to_dataset(name='std').to_netcdf(f'{SAVE_PATH}a52o_lai_lv_std.nc', compute=True)
p_value_lai_hv.to_netcdf(f'{SAVE_PATH}delta_lai_hv_std_p.nc', compute=True)
p_value_lai_lv.to_netcdf(f'{SAVE_PATH}delta_lai_lv_std_p.nc', compute=True)